# PyMOL workflow — Trabajo final de Bioinformática

Automatiza la parte PyMOL de la práctica CP3 (docking proteína-proteína) para los puntos **3** y **4** del trabajo final:

- **Prepare** — limpia un PDB (quita aguas/iones), renderiza cartoon + superficie.
- **Pair** — prepara receptor y ligando por separado para subirlos a ClusPro.
- **Analyze** — analiza un complejo post-docking: residuos de interfaz, RMSD vs nativo, render.

Cada sección es independiente. Edita la celda **Setup** con tus IDs/paths y corre las secciones que necesites.


## 1. Setup

Edita las variables de entrada aquí. Acepta IDs del PDB (4 caracteres) **o** paths locales a archivos `.pdb`.


In [ ]:
from pathlib import Path

import pymol_utils as pu

OUT = Path("./out").resolve()
OUT.mkdir(exist_ok=True)

# Entradas — cámbialas según tu sistema:
PROTEIN = "1ELT"          # un PDB ID o un path a un .pdb

RECEPTOR = "1ELT"          # para la sección Pair
LIGAND = "2KSY"

COMPLEX_PDB = None         # path al modelo de ClusPro (model.000.01.pdb)
NATIVE_PDB = None          # opcional: complejo cristalográfico de referencia
CHAIN_RECEPTOR = "A"       # cadenas en el complejo
CHAIN_LIGAND = "B"
INTERFACE_CUTOFF = 5.0     # Å

## 2. Prepare — un PDB

Descarga (si es ID), limpia y renderiza la proteína. Útil para el punto 2 (modelo 3D) y como paso previo al docking.


In [ ]:
raw = pu.fetch_or_load(PROTEIN, OUT)
clean = pu.clean_structure(raw, OUT / f"{Path(raw).stem}_clean.pdb")
png_ss = pu.render_cartoon(clean, OUT / f"{Path(raw).stem}_ss.png", color_by="ss")
png_chain = pu.render_cartoon(clean, OUT / f"{Path(raw).stem}_chain.png", color_by="chain")
png_surface = pu.render_surface(clean, OUT / f"{Path(raw).stem}_surface.png")

print("PDB crudo:     ", raw)
print("PDB limpio:    ", clean)
print("Cartoon (ss):  ", png_ss)
print("Cartoon (chain):", png_chain)
print("Superficie:    ", png_surface)

In [ ]:
from IPython.display import Image, display
for p in (png_ss, png_chain, png_surface):
    display(Image(str(p)))

## 3. Pair — receptor + ligando (pre-docking)

Prepara ambas proteínas por separado. Los `.pdb` limpios resultantes son los que subes a ClusPro.


In [ ]:
rec_raw = pu.fetch_or_load(RECEPTOR, OUT)
rec_clean = pu.clean_structure(rec_raw, OUT / f"receptor_{Path(rec_raw).stem}.pdb")
rec_png = pu.render_cartoon(rec_clean, OUT / f"receptor_{Path(rec_raw).stem}.png", color_by="ss")

lig_raw = pu.fetch_or_load(LIGAND, OUT)
lig_clean = pu.clean_structure(lig_raw, OUT / f"ligand_{Path(lig_raw).stem}.pdb")
lig_png = pu.render_cartoon(lig_clean, OUT / f"ligand_{Path(lig_raw).stem}.png", color_by="ss")

print("Receptor listo para ClusPro:", rec_clean)
print("Ligando listo para ClusPro: ", lig_clean)
print("\nSube ambos a https://cluspro.bu.edu/ y descarga `model.000.01.pdb`.")
print("Luego asígnalo a COMPLEX_PDB en la celda Setup y corre la sección 4.")

## 4. Analyze — complejo post-docking

Carga el complejo (de ClusPro o cualquier complejo dockeado), identifica los residuos de interfaz a < `INTERFACE_CUTOFF` Å, y opcionalmente lo alinea contra el complejo cristalográfico nativo.


In [ ]:
assert COMPLEX_PDB is not None, "Define COMPLEX_PDB en la celda Setup"

complex_path = pu.fetch_or_load(COMPLEX_PDB, OUT)
complex_clean = pu.clean_structure(complex_path, OUT / f"complex_{Path(complex_path).stem}.pdb")

iface = pu.interface_residues(complex_clean, CHAIN_RECEPTOR, CHAIN_LIGAND, cutoff=INTERFACE_CUTOFF)
csv_path = pu.save_interface_csv(iface, OUT / "interface_residues.csv")

print(f"Residuos de interfaz (<{INTERFACE_CUTOFF} Å):")
print(f"  cadena {CHAIN_RECEPTOR}: {(iface['side'] == 'A').sum()}")
print(f"  cadena {CHAIN_LIGAND}: {(iface['side'] == 'B').sum()}")
print(f"  CSV: {csv_path}")
iface

In [ ]:
complex_png = pu.render_cartoon(
    complex_clean,
    OUT / "complex_chains.png",
    color_by="chain",
)

highlights = [(r.chain, r.resi) for r in iface.itertuples()]
iface_png = pu.render_cartoon(
    complex_clean,
    OUT / "complex_interface.png",
    color_by="chain",
    highlight_residues=highlights,
)

from IPython.display import Image, display
for p in (complex_png, iface_png):
    display(Image(str(p)))

In [ ]:
if NATIVE_PDB is not None:
    native_raw = pu.fetch_or_load(NATIVE_PDB, OUT)
    native_clean = pu.clean_structure(native_raw, OUT / f"native_{Path(native_raw).stem}.pdb")
    info = pu.align_to_native(complex_clean, native_clean, out_png=OUT / "alignment.png")
    print(f"RMSD modelo vs nativo: {info['rmsd']:.3f} Å sobre {info['n_atoms']} átomos")
    display(Image(str(info['png'])))
else:
    print("NATIVE_PDB no definido — saltando comparación con estructura experimental.")

## Resumen

Todos los archivos generados quedan en `./out/`. Para el informe y el PPT (puntos 3 y 4 del trabajo final) puedes usar directamente:

- `*_clean.pdb` — estructuras limpias.
- `*_ss.png`, `*_chain.png`, `*_surface.png` — figuras de las proteínas individuales.
- `complex_chains.png`, `complex_interface.png` — figuras del complejo dockeado.
- `interface_residues.csv` — lista de residuos a <5 Å (punto 4: importancia de la interfaz).
- `alignment.png` — superposición modelo vs nativo (cyan = modelo, magenta = nativo).
